# SentVols — Production Annotator Test Suite
## Colab L4 GPU · Real Market News · All Annotators

This notebook tests every annotator in the `sentvols` library using **live financial news** pulled from Yahoo Finance via `yfinance`.  
It is designed to run in Google Colab (L4 GPU) as a production smoke-test.

### Working paradigm
> If a cell runs and returns an error: fix locally → `git commit` → `git push` → restart kernel → rerun from the failed cell.

### Annotators covered
| # | Annotator | Backend | Notes |
|---|-----------|---------|-------|
| 1 | `FinancialVADERAnnotator` | Rule-based LM lexicon | Always available, zero deps |
| 2 | `Annotator("vader")` | VADER via façade | High-level API |
| 3 | `FinancialLLMAnnotator` | HuggingFace `flan-t5-base` via `TransformersBackend` | CPU-compatible |
| 4 | `FinancialLLMAnnotator` | HuggingFace large model via `TransformersBackend` (GPU) | L4 GPU, ~7B |
| 5 | `Annotator("llm", ...)` | Same large model via façade | End-to-end |
| 6 | `Annotator(normalizer=...)` | VADER + LLM normalisation pre-processing | Full pipeline |

### Prerequisites
Run **Cell 2** first (installs/upgrades packages).  Then run cells sequentially.

In [1]:
## ── Cell 1 · Environment detection ─────────────────────────────────────────
import os
import sys
import subprocess

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

IS_GPU = False
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=5,
    )
    gpu_name = result.stdout.strip()
    IS_GPU = bool(gpu_name)
    if IS_GPU:
        print(f"GPU detected : {gpu_name}")
except Exception:
    pass

if not IS_GPU:
    print("No GPU detected — CPU-only mode.")

DEVICE = "cuda" if IS_GPU else "cpu"
print(f"Running in Colab : {IN_COLAB}")
print(f"Python version   : {sys.version.split()[0]}")
print(f"Device           : {DEVICE}")

GPU detected : NVIDIA L4
Running in Colab : True
Python version   : 3.12.13
Device           : cuda


In [2]:
## ── Cell 2 · Make sentvols importable + install lightweight deps ────────────
# Strategy: clone the repo and inject the path — avoids pip-install side-effects
# (pip can trigger a Colab kernel restart when torch/CUDA packages are touched).

import sys, subprocess, pathlib

def _pip(*args):
    """Run pip as a subprocess; print stderr so failures are visible."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args],
        capture_output=True, text=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip()[-1000:])
    if result.returncode != 0:
        print(result.stderr.strip()[-2000:])
        raise RuntimeError(f"pip {list(args)} failed (exit {result.returncode})")

# ── Make sentvols importable ─────────────────────────────────────────────────
try:
    import sentvols  # noqa: F401
    print("sentvols already importable:", sentvols.__file__)
except ImportError:
    if IN_COLAB:
        REPO_DIR = "/content/sentvols_repo"
        if not pathlib.Path(REPO_DIR).exists():
            print("Cloning sentvols repo …")
            subprocess.run(
                ["git", "clone", "--depth=1",
                 "https://github.com/Seydifa/sentvols.git", REPO_DIR],
                check=True,
            )
            print("Cloned.")
        else:
            print("Repo already cloned — pulling latest …")
            subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
        # Inject path so Python can find the package without pip
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        import sentvols  # noqa: F401
        print("sentvols importable via sys.path:", sentvols.__file__)
    else:
        raise RuntimeError("sentvols not found. Activate your local venv and try again.")

# ── Install lightweight deps that are NOT pre-installed on Colab ─────────────
# torch / accelerate / bitsandbytes are already on Colab L4 — upgrading them
# triggers a kernel restart.  We only add packages that are genuinely missing.

print("Installing / upgrading lightweight packages …")
_pip(
    "install", "-q", "--upgrade",
    "yfinance>=1.0",
    "sentencepiece",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm",
    "vaderSentiment",
    "optuna",
    "lightgbm",
    "scikit-learn",
)
print("All dependencies ready.")

Repo already cloned — pulling latest …
sentvols importable via sys.path: /content/sentvols_repo/sentvols/__init__.py
Installing / upgrading lightweight packages …
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 152.3 MB/s eta 0:00:00
All dependencies ready.


In [3]:
## ── Cell 3 · Imports & configuration ────────────────────────────────────────
import warnings, datetime, json

import pandas as pd
import matplotlib
matplotlib.use("Agg")           # non-interactive backend — no display needed
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yfinance as yf
from tqdm.auto import tqdm

from sentvols.utils import (
    FinancialVADERAnnotator,
    FinancialLLMAnnotator,
    Annotator,
    TransformersBackend,
    NormalizationResult,
    NormalizerBackend,
    FinancialTextNormalizer,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_style("whitegrid")

# ── Tickers to analyse — diversified across sectors ─────────────────────────
TICKERS = ["AAPL", "MSFT", "NVDA", "JPM", "XOM", "TSLA"]

# ── How many news items to fetch per ticker ──────────────────────────────────
NEWS_PER_TICKER = 10

print("Imports OK.")
print(f"Tickers : {TICKERS}")
print(f"News per ticker : {NEWS_PER_TICKER}")

Imports OK.
Tickers : ['AAPL', 'MSFT', 'NVDA', 'JPM', 'XOM', 'TSLA']
News per ticker : 10


In [4]:
## ── Cell 4 · Fetch live financial news from Yahoo Finance ────────────────────
# yfinance ≥1.0 nests the article under item["content"]. We strip HTML
# from summaries and fallback gracefully if fields are missing.

import re as _re

def _strip_html(text: str) -> str:
    """Remove HTML tags from a string."""
    return _re.sub(r"<[^>]+>", " ", text or "").strip()


def fetch_news(tickers: list[str], max_per_ticker: int = 10) -> pd.DataFrame:
    """Fetch recent news headlines + summaries from Yahoo Finance.

    Returns a DataFrame with columns:
        ticker, pub_date, title, summary, url
    """
    rows = []
    for ticker in tqdm(tickers, desc="Fetching news"):
        try:
            t = yf.Ticker(ticker)
            news_items = t.news or []
        except Exception as exc:
            print(f"  [WARN] Could not fetch news for {ticker}: {exc}")
            continue

        count = 0
        for item in news_items:
            if count >= max_per_ticker:
                break
            content = item.get("content", item)  # v1.x wraps in "content"
            title   = _strip_html(content.get("title", ""))
            summary = _strip_html(content.get("summary", content.get("description", "")))
            pub_raw = content.get("pubDate", content.get("providerPublishTime", ""))
            url     = ""
            # Extract canonical url safely
            canonical = content.get("canonicalUrl", {})
            if isinstance(canonical, dict):
                url = canonical.get("url", "")
            if not url:
                url = content.get("url", "")

            if not title:
                continue

            rows.append({
                "ticker"  : ticker,
                "pub_date": str(pub_raw)[:19],
                "title"   : title,
                "summary" : summary[:300] if summary else "",
                "url"     : url,
            })
            count += 1

    df = pd.DataFrame(rows)
    print(f"\nFetched {len(df)} articles across {df['ticker'].nunique()} tickers.")
    return df


df_news = fetch_news(TICKERS, max_per_ticker=NEWS_PER_TICKER)
df_news.head(10)

Fetching news:   0%|          | 0/6 [00:00<?, ?it/s]


Fetched 60 articles across 6 tickers.


,ticker,pub_date,title,summary,url
0,AAPL,2026-04-06T16:00:00,OpenAI IPO scrutiny builds after acquiring tech podcast TBPN,Morning Brief Anchor Julie Hyman and Yahoo Finance Head of News Myles Udland discuss r...,https://finance.yahoo.com/video/openai-ipo-scrutiny-builds-after-acquiring-tech-podcas...
1,AAPL,2026-04-06T21:19:03,Here’s Why Warren Buffett and Ken Griffin Love Apple (AAPL),We just covered the 10 Best Stocks to Buy According to Billionaire Ken Griffin. Apple ...,https://finance.yahoo.com/markets/stocks/articles/why-warren-buffett-ken-griffin-21190...
2,AAPL,2026-04-06T20:24:10,Stock Market Today: Dow Up As Trump Offers This Olive Branch; AI Play Eyes Entry (Live...,The Dow Jones index and Apple rose on the stock market today. President Donald Trump o...,https://www.investors.com/market-trend/stock-market-today/dow-jones-sp500-nasdaq-trump...
3,AAPL,2026-04-06T18:59:55,Apple Faces Questions in Amazon-Globalstar Talks,Potential deal could reshape satellite strategy and impact Apple's existing partnership.,https://finance.yahoo.com/markets/stocks/articles/apple-faces-questions-amazon-globals...
4,AAPL,2026-04-06T17:49:53,Amazon Globalstar Deal Puts Apple In Middle,Satellite access could complicate existing ties,https://finance.yahoo.com/markets/stocks/articles/amazon-globalstar-deal-puts-apple-17...
5,AAPL,2026-04-06T17:38:05,Apple is taking its App Store fight to the Supreme Court — again,"Apple plans to ask the Supreme Court to review its App Store fight with Epic Games, as...",https://finance.yahoo.com/sectors/technology/articles/apple-plans-supreme-court-appeal...
6,AAPL,2026-04-06T16:37:00,"Apple MacBook Neo could expand market reach, says Bank of America","Apple Inc (NASDAQ:AAPL, XETRA:APC) is positioning itself to capture a broader segment ...",https://www.proactiveinvestors.com/companies/news/1090053/apple-macbook-neo-could-expa...
7,AAPL,2026-04-06T16:31:58,Why Intel and AMD Stocks Are Rising Today,"Intel, Advanced Micro Devices Jump on Strong Server CPU Demand",https://finance.yahoo.com/markets/stocks/articles/why-intel-amd-stocks-rising-16315866...
8,AAPL,2026-04-06T16:26:00,AI Is Squeezing Memory Supplies. How Apple’s Strategy Can Boost Its Stock.,Apple is using its product environment to play the long game. Seaport analyst Jay Gold...,https://www.barrons.com/articles/ai-apple-stock-price-4656798f?siteid=yhoof2&yptr=yahoo
9,AAPL,2026-04-06T15:29:43,Apple’s Record $143 Billion Quarter Exposes Just How Far Tesla Has Fallen,"Apple (NASDAQ:AAPL) and Tesla (NASDAQ:TSLA) both reported earnings this year, and the ...",https://247wallst.com/investing/2026/04/06/apples-record-143-billion-quarter-exposes-j...


In [5]:
## ── Cell 5 · Fetch recent price data (ground-truth returns) ─────────────────
# We fetch 30 trading-day returns so we can visually compare sentiment
# direction against price movement — a basic reality check.

def fetch_price_returns(tickers: list[str], period: str = "1mo") -> pd.DataFrame:
    """Fetch recent close prices and compute daily returns."""
    frames = []
    for ticker in tqdm(tickers, desc="Fetching prices"):
        try:
            hist = yf.Ticker(ticker).history(period=period)
            if hist.empty:
                continue
            hist = hist[["Close"]].copy()
            hist.index = hist.index.tz_localize(None)  # strip tz for simplicity
            hist.columns = ["close"]
            hist["ticker"] = ticker
            hist["return"] = hist["close"].pct_change()
            frames.append(hist.reset_index())
        except Exception as exc:
            print(f"  [WARN] Price fetch failed for {ticker}: {exc}")

    df_prices = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    df_prices.columns = [c.lower() for c in df_prices.columns]
    df_prices = df_prices.rename(columns={"date": "trade_date"})
    print(f"Fetched {len(df_prices)} price rows.")
    return df_prices


df_prices = fetch_price_returns(TICKERS)

# Latest return per ticker — useful for later comparison
latest_returns = (
    df_prices.dropna(subset=["return"])
    .sort_values("trade_date")
    .groupby("ticker")
    .last()["return"]
    .reset_index()
    .rename(columns={"return": "latest_1d_return"})
)
print("\n30-day latest 1-day returns:")
print(latest_returns.to_string(index=False))

Fetching prices:   0%|          | 0/6 [00:00<?, ?it/s]

Fetched 120 price rows.

30-day latest 1-day returns:
ticker  latest_1d_return
  AAPL            0.0115
   JPM            0.0029
  MSFT           -0.0016
  NVDA            0.0014
  TSLA           -0.0215
   XOM            0.0167


---
## Annotator 1 — `FinancialVADERAnnotator` (rule-based LM lexicon)

Zero dependencies.  The workhorse lexicon annotator with the full Loughran-McDonald dictionary.

In [ ]:
## ── Cell 6 · FinancialVADERAnnotator — construction & sanity checks ──────────
vader_ann = FinancialVADERAnnotator(
    pos_threshold=0.05,
    neg_threshold=-0.05,
    phrase_weight=0.3,
)

# ── Hard-coded financial sentences with expected polarity ────────────────────
GOLDEN_SET = [
    # (text, expected_label)
    ("Apple beats earnings estimates, raises guidance",          "positif"),
    ("The company announced record revenue and a dividend hike", "positif"),
    ("Stock buyback programme worth $10 billion approved",       "positif"),
    ("The company reported a loss and issued a profit warning",  "négatif"),
    ("Chapter 11 bankruptcy protection filing announced",        "négatif"),
    ("Credit downgrade from investment grade to junk",          "négatif"),
    ("Analysts do not expect the firm to file for bankruptcy",   "neutre"),
    ("The board met to discuss quarterly operational matters",   "neutre"),
]

# ── Score and verify ─────────────────────────────────────────────────────────
results = []
for text, expected in GOLDEN_SET:
    annotation = vader_ann.annotate(text)
    ok = annotation["label"] == expected
    results.append({
        "text"    : text[:75],
        "expected": expected,
        "label"   : annotation["label"],
        "score"   : annotation["score"],
        "OK"      : "✓" if ok else "✗",
    })

df_golden = pd.DataFrame(results)
print(df_golden.to_string(index=False))

n_pass = (df_golden["OK"] == "✓").sum()
n_total = len(df_golden)
print(f"\nGolden set: {n_pass}/{n_total} passed")
# Accept 7/8: deep long-range negation (>3 tokens away) is a known
# lexicon-based limit — "do not expect ... file for bankruptcy" requires
# phrase-table coverage added in CONTEXT_PHRASES to reliably flip.
assert n_pass >= n_total - 1, (
    f"FinancialVADERAnnotator failed {n_total - n_pass} golden-set items — "
    "check annotators.py or the LM lexicon."
)

                                                    text expected   label   score OK
         Apple beats earnings estimates, raises guidance  positif positif  0.6705  ✓
The company announced record revenue and a dividend hike  positif positif  0.5700  ✓
      Stock buyback programme worth $10 billion approved  positif positif  1.0000  ✓
 The company reported a loss and issued a profit warning  négatif négatif -1.0000  ✓
       Chapter 11 bankruptcy protection filing announced  négatif négatif -1.0000  ✓
          Credit downgrade from investment grade to junk  négatif négatif -0.7500  ✓
  Analysts do not expect the firm to file for bankruptcy   neutre négatif -0.5423  ✗
  The board met to discuss quarterly operational matters   neutre  neutre  0.0258  ✓

Golden set: 7/8 passed


AssertionError: FinancialVADERAnnotator failed 1 golden-set items — check annotators.py or the LM lexicon.

In [ ]:
## ── Cell 7 · FinancialVADERAnnotator — score live news ──────────────────────
df_vader = df_news.copy()
df_vader["score"] = df_vader["title"].apply(vader_ann.score)
df_vader["label"] = df_vader["score"].apply(vader_ann.label)

print(f"Annotated {len(df_vader)} headlines with FinancialVADERAnnotator.")
print(df_vader[["ticker", "title", "score", "label"]].head(15).to_string(index=False))

# Distribution per ticker
print("\n── Sentiment distribution per ticker ──")
print(
    df_vader.groupby(["ticker", "label"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
## ── Cell 8 · FinancialVADERAnnotator — explain one headline per ticker ───────
print("── Score explanation (top word / phrase contributions) ──\n")
for ticker in TICKERS:
    sample = df_vader[df_vader["ticker"] == ticker].iloc[0]
    exp = vader_ann.explain(sample["title"])
    top_words   = [f"{w['word']}({w['valence']:+.1f})"
                   for w in exp["word_hits"][:3]]
    top_phrases = [f"'{p['phrase']}'({p['adjustment']:+.1f})"
                   for p in exp["phrase_hits"][:2]]
    print(f"[{ticker}] {sample['title'][:70]}")
    print(f"  score={exp['final_score']:+.4f}  label={exp['label']}")
    if top_words:
        print(f"  words  : {', '.join(top_words)}")
    if top_phrases:
        print(f"  phrases: {', '.join(top_phrases)}")
    print()

In [ ]:
## ── Cell 9 · FinancialVADERAnnotator — visualise sentiment vs return ─────────
# Merge mean sentiment score per ticker with latest 1-day price return.

mean_sent = df_vader.groupby("ticker")["score"].mean().reset_index()
mean_sent.columns = ["ticker", "mean_vader_score"]
df_compare = mean_sent.merge(latest_returns, on="ticker", how="left")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("FinancialVADERAnnotator — Live News Sentiment", fontsize=13, fontweight="bold")

# Bar: mean sentiment per ticker
colors = ["#2ecc71" if s > 0.05 else "#e74c3c" if s < -0.05 else "#95a5a6"
          for s in df_compare["mean_vader_score"]]
axes[0].bar(df_compare["ticker"], df_compare["mean_vader_score"], color=colors)
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].axhline(0.05,  color="#2ecc71", linewidth=0.8, linestyle=":")
axes[0].axhline(-0.05, color="#e74c3c", linewidth=0.8, linestyle=":")
axes[0].set_title("Mean VADER+LM score per ticker")
axes[0].set_ylabel("Score [-1, +1]")
axes[0].set_xlabel("Ticker")

# Scatter: mean sentiment vs latest return
axes[1].scatter(df_compare["mean_vader_score"], df_compare["latest_1d_return"] * 100,
                s=80, zorder=3)
for _, row in df_compare.iterrows():
    axes[1].annotate(row["ticker"],
                     (row["mean_vader_score"], row["latest_1d_return"] * 100),
                     textcoords="offset points", xytext=(6, 4), fontsize=9)
axes[1].axhline(0, color="black", linewidth=0.6, linestyle="--")
axes[1].axvline(0, color="black", linewidth=0.6, linestyle="--")
axes[1].set_title("Sentiment score vs latest 1-day return (%)")
axes[1].set_xlabel("Mean VADER+LM score")
axes[1].set_ylabel("Latest 1-day return (%)")

plt.tight_layout()
plt.show()
print(df_compare.to_string(index=False))

---
## Annotator 2 — `Annotator("vader")` high-level façade

Same lexicon engine, accessed through the user-facing `Annotator` class.

In [ ]:
## ── Cell 10 · Annotator("vader") — façade wrapper ───────────────────────────
facade_ann = Annotator(backend="vader", pos_threshold=0.05, neg_threshold=-0.05)

# Verify it wraps FinancialVADERAnnotator
assert isinstance(facade_ann.inner_annotator, FinancialVADERAnnotator), (
    "Annotator('vader') should wrap FinancialVADERAnnotator"
)

# Score the full live set via batch API
facade_scores = facade_ann.score_batch(df_news["title"].tolist(), workers=4)
facade_labels = [facade_ann.label(s) for s in facade_scores]

df_facade = df_news.copy()
df_facade["score"] = facade_scores
df_facade["label"] = facade_labels

# Scores must exactly match FinancialVADERAnnotator (same engine)
vader_scores = vader_ann.score_batch(df_news["title"].tolist(), workers=4)
assert facade_scores == vader_scores, (
    "Annotator('vader') scores diverge from FinancialVADERAnnotator — "
    "possible regression in façade delegation."
)
print("Façade scores identical to direct annotator: ✓")

# article-level API on a longer sample text
long_article = (
    "Apple reported quarterly earnings that beat analyst expectations by a wide margin. "
    "Revenue rose 8% year-over-year driven by strong iPhone and Services growth. "
    "The board approved a $90 billion share buyback and raised the quarterly dividend. "
    "However, management issued a cautious outlook for next quarter citing macroeconomic "
    "uncertainty and warned that supply chain constraints may weigh on margins."
)
article_result = facade_ann.annotate_article(long_article)
print(f"\nArticle-level annotation : {article_result}")
print(f"  (score_article with decay=0.9 weights lead sentences more heavily)")

---
## Annotator 3 — `FinancialLLMAnnotator` + `TransformersBackend` (small model, CPU-safe)

Uses `google/flan-t5-base` (~250 MB).  Works on CPU — no GPU required.  
Good sanity-check before loading a large model.

In [ ]:
## ── Cell 11 · TransformersBackend (flan-t5-base) + FinancialLLMAnnotator ─────
# flan-t5-base is a seq2seq model.  TransformersBackend uses AutoModelForSeq2SeqLM
# internally — it works regardless of whether the pipeline registry supports
# text2text-generation in the current transformers version.

SMALL_MODEL = "google/flan-t5-base"

print(f"Loading TransformersBackend with model={SMALL_MODEL!r} on device={DEVICE!r} …")
small_backend = TransformersBackend(model=SMALL_MODEL, device=DEVICE, max_new_tokens=64)

# Verify NormalizerBackend protocol
assert isinstance(small_backend, NormalizerBackend), (
    "TransformersBackend does not satisfy NormalizerBackend protocol — "
    "check that it has .model, .reasoning_available, and .call()"
)

llm_ann_small = FinancialLLMAnnotator(
    backend=small_backend,
    pos_threshold=0.05,
    neg_threshold=-0.05,
)
print(f"FinancialLLMAnnotator built: {llm_ann_small!r}")

In [ ]:
## ── Cell 12 · flan-t5-base — score golden set & live headlines ───────────────
# We test a subset to keep inference time reasonable on CPU.

SAMPLE_SIZE = min(12, len(df_news))
df_sample = df_news.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Scoring {SAMPLE_SIZE} headlines with {SMALL_MODEL} …")
small_results = []
for _, row in tqdm(df_sample.iterrows(), total=SAMPLE_SIZE, desc="LLM scoring (small)"):
    ann_result = llm_ann_small.annotate(row["title"])
    small_results.append({
        "ticker": row["ticker"],
        "title" : row["title"][:70],
        "llm_score": ann_result["score"],
        "llm_label": ann_result["label"],
        "vader_score": vader_ann.score(row["title"]),
        "vader_label": vader_ann.label(vader_ann.score(row["title"])),
    })

df_small_llm = pd.DataFrame(small_results)
print(df_small_llm[["ticker", "title", "llm_score", "llm_label",
                     "vader_score", "vader_label"]].to_string(index=False))

# Agreement rate between flan-t5 and VADER
agree = (df_small_llm["llm_label"] == df_small_llm["vader_label"]).mean()
print(f"\nLabel agreement (small LLM vs VADER): {agree:.0%}")

---
## Annotator 4 — `FinancialLLMAnnotator` + large model (L4 GPU)

Uses `google/flan-t5-xl` (~3B params, ~5 GB VRAM) when a GPU is available.  
Falls back to `flan-t5-large` (~800 MB) on CPU if no GPU is detected.

> **Colab L4 note**: the L4 has 24 GB VRAM — more than enough for flan-t5-xl in float16.

In [ ]:
## ── Cell 13 · Load large Transformers model ─────────────────────────────────
# GPU: flan-t5-xl (3B)  |  CPU fallback: flan-t5-large (~800 MB)
#
# TransformersBackend loads the model lazily on first .call() — this cell
# only instantiates the backend object to validate construction.
# The actual model download + load happens in Cell 14.

LARGE_MODEL = "google/flan-t5-xl" if IS_GPU else "google/flan-t5-large"
print(f"Large model selected : {LARGE_MODEL}  (device={DEVICE})")

large_backend = TransformersBackend(
    model=LARGE_MODEL,
    device=DEVICE,
    max_new_tokens=64,
)

llm_ann_large = FinancialLLMAnnotator(
    backend=large_backend,
    pos_threshold=0.05,
    neg_threshold=-0.05,
)
print(f"FinancialLLMAnnotator (large): {llm_ann_large!r}")
print("Model will be downloaded and loaded on first inference call (Cell 14).")

In [ ]:
## ── Cell 14 · Large LLM — score golden set + live news ─────────────────────
# First call triggers model download & load — may take a few minutes on Colab.
# Subsequent calls in this session are fast (model is cached on the backend).

print(f"Warming up {LARGE_MODEL} (first call loads the model) …")
_warmup_score = llm_ann_large.score("Apple beats estimates")
print(f"  Warm-up score: {_warmup_score:+.4f}  ✓ model loaded")

# Golden set with large model
print(f"\n── Golden set — {LARGE_MODEL} ──")
large_golden = []
for text, expected in GOLDEN_SET:
    ann_result = llm_ann_large.annotate(text)
    ok = ann_result["label"] == expected
    large_golden.append({
        "text"    : text[:65],
        "expected": expected,
        "label"   : ann_result["label"],
        "score"   : ann_result["score"],
        "OK"      : "✓" if ok else "✗",
    })
df_large_golden = pd.DataFrame(large_golden)
print(df_large_golden.to_string(index=False))
n_pass_large = (df_large_golden["OK"] == "✓").sum()
print(f"\nGolden set (large LLM): {n_pass_large}/{len(df_large_golden)}")

# Live news
print(f"\n── Live news — {LARGE_MODEL} ──")
large_live = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample),
                   desc=f"LLM scoring ({LARGE_MODEL.split('/')[-1]})"):
    ann_result = llm_ann_large.annotate(row["title"])
    large_live.append({
        "ticker"    : row["ticker"],
        "title"     : row["title"][:70],
        "large_score": ann_result["score"],
        "large_label": ann_result["label"],
    })
df_large_live = pd.DataFrame(large_live)
print(df_large_live.to_string(index=False))

---
## Annotator 5 — `Annotator("llm", ...)` high-level façade with large model

The `Annotator` façade wraps any LLM backend.  This tests the full `backend="llm"` path end-to-end.

In [ ]:
## ── Cell 15 · Annotator("llm") façade with large backend ───────────────────
# Reuse the already-loaded large_backend (model stays in GPU memory).
facade_llm = Annotator(
    backend="llm",
    llm_backend=large_backend,
    pos_threshold=0.05,
    neg_threshold=-0.05,
)

assert isinstance(facade_llm.inner_annotator, FinancialLLMAnnotator), (
    "Annotator('llm') should wrap FinancialLLMAnnotator"
)

# Verify that the façade delegates to the same underlying backend
test_text = "Company announces record dividend hike"
facade_score = facade_llm.score(test_text)
direct_score = llm_ann_large.score(test_text)
assert abs(facade_score - direct_score) < 1e-9, (
    f"Score mismatch between Annotator('llm') and direct FinancialLLMAnnotator: "
    f"{facade_score} vs {direct_score}"
)
print(f"Façade score == direct score: {facade_score:+.4f}  ✓")

# Batch scoring on full live news set via the façade
print(f"\nBatch-annotating {len(df_news)} headlines via Annotator('llm') …")
facade_llm_scores = facade_llm.score_batch(df_news["title"].tolist(), workers=1)
df_facade_llm = df_news.copy()
df_facade_llm["score"] = facade_llm_scores
df_facade_llm["label"] = [facade_llm.label(s) for s in facade_llm_scores]
print(df_facade_llm[["ticker", "title", "score", "label"]].head(12).to_string(index=False))

---
## Annotator 6 — `Annotator` with VADER + LLM normalisation pre-processing

The normaliser rewrites complex texts (sarcasm, deep negation, jargon) into literal financial facts **before** VADER scores them.  This is the full production pipeline.

In [ ]:
## ── Cell 16 · VADER + LLM normaliser (full pipeline) ───────────────────────
# We build a FinancialTextNormalizer backed by the large Transformers model,
# then wrap it in Annotator(backend="vader", normalizer=...).
# The normaliser fires only for texts longer than normalize_threshold chars.

from sentvols.utils import FinancialTextNormalizer

norm = FinancialTextNormalizer(backend=large_backend)

pipeline_ann = Annotator(
    backend="vader",
    normalizer=norm,
    normalize_threshold=80,    # headlines shorter than 80 chars bypass the LLM
    normalize_mode="rewrite",  # resolve sarcasm / long-range negation
    pos_threshold=0.05,
    neg_threshold=-0.05,
)
print(f"Pipeline annotator: {pipeline_ann!r}")

# Tricky sarcasm / deep-negation cases designed to fool plain VADER
TRICKY_CASES = [
    (
        "Oh great, another record quarter — record losses, that is.  "
        "Management helpfully reminded investors that burning through cash at "
        "twice the rate is part of the plan.",
        "négatif",   # sarcasm — should become negative after rewrite
    ),
    (
        "Analysts do not expect the firm to miss its revenue targets given "
        "the strong order book and raised guidance.",
        "positif",   # double negation — should score positive
    ),
    (
        "Apple reported stronger earnings and raised the dividend despite "
        "headwinds in the Chinese market and a weakening consumer environment.",
        "positif",   # positive core with hedging noise
    ),
]

print("\n── Tricky-case rewriting pipeline ──")
for text, expected in TRICKY_CASES:
    # Inspect what the normaliser produces
    norm_result = norm.normalize(text, mode="rewrite")
    ann_result  = pipeline_ann.annotate(text)
    ok = ann_result["label"] == expected
    print(f"INPUT   : {text[:90]}...")
    print(f"REWRITE : {norm_result.normalized_text[:90]}")
    print(f"RESULT  : score={ann_result['score']:+.4f}  label={ann_result['label']}  "
          f"expected={expected}  {'✓' if ok else '✗'}")
    print()

In [ ]:
## ── Cell 17 · Pipeline — normalise & score live summaries ───────────────────
# Use article summaries (they are longer, so the normaliser is triggered more
# often by the 80-char threshold).

df_summary = df_news[df_news["summary"].str.len() > 0].copy().reset_index(drop=True)

pipeline_results = []
for _, row in tqdm(df_summary.iterrows(), total=len(df_summary),
                   desc="Pipeline (VADER + normaliser)"):
    text = row["summary"]
    ann_result = pipeline_ann.annotate(text)
    # Also score raw text with plain VADER for delta analysis
    raw_score  = vader_ann.score(text)
    pipeline_results.append({
        "ticker"        : row["ticker"],
        "summary"       : text[:60],
        "pipeline_score": ann_result["score"],
        "pipeline_label": ann_result["label"],
        "raw_vader_score": raw_score,
        "score_delta"   : ann_result["score"] - raw_score,
    })

df_pipeline = pd.DataFrame(pipeline_results)
print(df_pipeline.to_string(index=False))
print(f"\nMean |score_delta| (normaliser effect): "
      f"{df_pipeline['score_delta'].abs().mean():.4f}")

---
## Cross-annotator comparison

Side-by-side view of every annotator's output on the same live headlines.  
Also includes a visual breakdown of label agreement / disagreement.

In [ ]:
## ── Cell 18 · Multi-annotator comparison table ───────────────────────────────
# Build a unified DataFrame on the same df_sample rows.

comparison_rows = []
for _, row in df_sample.iterrows():
    title = row["title"]
    v_score  = vader_ann.score(title)
    sl_score = llm_ann_small.score(title)
    ll_score = llm_ann_large.score(title)
    comparison_rows.append({
        "ticker"       : row["ticker"],
        "title"        : title[:70],
        "vader_score"  : v_score,
        "vader_label"  : vader_ann.label(v_score),
        "small_llm_score" : sl_score,
        "small_llm_label" : llm_ann_small.label(sl_score),
        "large_llm_score" : ll_score,
        "large_llm_label" : llm_ann_large.label(ll_score),
    })

df_comparison = pd.DataFrame(comparison_rows)

# Highlight whether all three models agree
def _all_agree(row):
    labels = {row["vader_label"], row["small_llm_label"], row["large_llm_label"]}
    return "✓" if len(labels) == 1 else "✗"

df_comparison["all_agree"] = df_comparison.apply(_all_agree, axis=1)
agree_rate = (df_comparison["all_agree"] == "✓").mean()

print(df_comparison[[
    "ticker", "title",
    "vader_label", "small_llm_label", "large_llm_label", "all_agree"
]].to_string(index=False))
print(f"\n3-way agreement rate: {agree_rate:.0%}")

In [ ]:
## ── Cell 19 · Comparison visualisations ─────────────────────────────────────
LABEL_ORDER = ["positif", "neutre", "négatif"]
LABEL_COLOR = {"positif": "#2ecc71", "neutre": "#95a5a6", "négatif": "#e74c3c"}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
models = [
    ("vader_label",     "VADER + LM"),
    ("small_llm_label", f"LLM small\n({SMALL_MODEL.split('/')[-1]})"),
    ("large_llm_label", f"LLM large\n({LARGE_MODEL.split('/')[-1]})"),
]

for ax, (col, title) in zip(axes, models):
    counts = df_comparison[col].value_counts()
    for i, lbl in enumerate(LABEL_ORDER):
        ax.bar(i, counts.get(lbl, 0), color=LABEL_COLOR[lbl])
    ax.set_xticks(range(len(LABEL_ORDER)))
    ax.set_xticklabels(LABEL_ORDER, rotation=20)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Count" if ax == axes[0] else "")

fig.suptitle("Sentiment label distribution per annotator (live news sample)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Score correlation heatmap
df_scores = df_comparison[["vader_score", "small_llm_score", "large_llm_score"]].copy()
df_scores.columns = ["VADER+LM", f"Small LLM\n({SMALL_MODEL.split('/')[-1]})",
                     f"Large LLM\n({LARGE_MODEL.split('/')[-1]})"]

corr = df_scores.corr()
fig2, ax2 = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            ax=ax2, square=True, linewidths=0.5)
ax2.set_title("Score correlation between annotators", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Lexicon & runtime extension tests

These cells verify the runtime extension API: `add_words`, `add_phrases`, `remove_words`, threshold setters.

In [ ]:
## ── Cell 20 · Runtime lexicon / threshold extension API ─────────────────────
ext_ann = FinancialVADERAnnotator()

# ── 1. add_words ─────────────────────────────────────────────────────────────
ext_ann.add_words({"moonshot": 3.0, "delist": -3.0})
score_moon  = ext_ann.score("Company launches moonshot strategy")
score_dlist = ext_ann.score("Regulator moves to delist the stock")
assert score_moon  > 0.05, f"Expected positive after adding 'moonshot', got {score_moon}"
assert score_dlist < -0.05, f"Expected negative after adding 'delist', got {score_dlist}"
print(f"add_words: moonshot={score_moon:+.4f} ✓   delist={score_dlist:+.4f} ✓")

# ── 2. remove_words ───────────────────────────────────────────────────────────
ext_ann.remove_words(["moonshot"])
score_moon_after = ext_ann.score("Company launches moonshot strategy")
print(f"remove_words: score after removal={score_moon_after:+.4f} "
      f"(should differ from {score_moon:+.4f} ✓)")

# ── 3. add_phrases ────────────────────────────────────────────────────────────
ext_ann.add_phrases({"reverse split": -2.0, "spinoff": 1.5})
score_rs = ext_ann.score("Board approves reverse split at 1-for-10")
assert score_rs < -0.05, f"Expected negative for 'reverse split', got {score_rs}"
print(f"add_phrases: 'reverse split' score={score_rs:+.4f} ✓")

# ── 4. threshold setters ─────────────────────────────────────────────────────
ext_ann.pos_threshold = 0.2
ext_ann.neg_threshold = -0.2
lbl_at_015 = ext_ann.label(0.15)  # should be neutre after tightening
assert lbl_at_015 == "neutre", f"Expected 'neutre' at 0.15 with threshold 0.2, got {lbl_at_015}"
ext_ann.pos_threshold = 0.05
ext_ann.neg_threshold = -0.05
lbl_restored = ext_ann.label(0.15)  # should be positif again
assert lbl_restored == "positif", f"Expected 'positif' after restoring threshold"
print(f"Threshold setters: label(0.15) tightened={lbl_at_015} restored={lbl_restored} ✓")

# ── 5. lexicon_snapshot ───────────────────────────────────────────────────────
snap = ext_ann.lexicon_snapshot
assert "delist" in snap, "'delist' should still be in lexicon"
assert snap["delist"] == -3.0
print(f"lexicon_snapshot: delist={snap['delist']} ✓")

print("\nAll runtime extension tests passed ✓")

---
## Final report — summary table + sentiment timeline

In [ ]:
## ── Cell 21 · Annotate full news set with all three annotators ──────────────
# Build a combined DataFrame with scores from all annotators for every headline.

print("Annotating full news set with all annotators …")
df_full = df_news.copy()
df_full["vader_score"]     = df_full["title"].apply(vader_ann.score)
df_full["vader_label"]     = df_full["vader_score"].apply(vader_ann.label)

small_scores = []
large_scores = []
for title in tqdm(df_full["title"], desc="Dual LLM scoring"):
    small_scores.append(llm_ann_small.score(title))
    large_scores.append(llm_ann_large.score(title))

df_full["small_llm_score"] = small_scores
df_full["small_llm_label"] = [llm_ann_small.label(s) for s in small_scores]
df_full["large_llm_score"] = large_scores
df_full["large_llm_label"] = [llm_ann_large.label(s) for s in large_scores]

# Ensemble: mean of all three scores as a robust signal
df_full["ensemble_score"] = (
    df_full["vader_score"] + df_full["small_llm_score"] + df_full["large_llm_score"]
) / 3.0
df_full["ensemble_label"] = df_full["ensemble_score"].apply(
    lambda s: "positif" if s >= 0.05 else ("négatif" if s <= -0.05 else "neutre")
)

print(f"\nFull annotated set ({len(df_full)} rows):")
print(df_full[[
    "ticker", "title", "vader_label", "small_llm_label",
    "large_llm_label", "ensemble_score", "ensemble_label"
]].head(20).to_string(index=False))

In [ ]:
## ── Cell 22 · Final report — ensemble sentiment vs price return ──────────────
mean_ensemble = (
    df_full.groupby("ticker")["ensemble_score"].mean().reset_index()
    .rename(columns={"ensemble_score": "mean_ensemble"})
)
df_final = mean_ensemble.merge(latest_returns, on="ticker", how="left")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Ensemble Sentiment (VADER + Small LLM + Large LLM) vs Market Returns",
             fontsize=12, fontweight="bold")

# Grouped sentiment bars per ticker
x = range(len(df_final))
axes[0].bar(x, df_final["mean_ensemble"],
            color=["#2ecc71" if s > 0.05 else "#e74c3c" if s < -0.05 else "#95a5a6"
                   for s in df_final["mean_ensemble"]])
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(df_final["ticker"].tolist())
axes[0].axhline(0,    color="black",   linewidth=0.8, linestyle="--")
axes[0].axhline(0.05, color="#2ecc71", linewidth=0.8, linestyle=":")
axes[0].axhline(-0.05,color="#e74c3c", linewidth=0.8, linestyle=":")
axes[0].set_title("Mean ensemble score per ticker")
axes[0].set_ylabel("Score [-1, +1]")

axes[1].scatter(df_final["mean_ensemble"], df_final["latest_1d_return"] * 100,
                s=80, zorder=3)
for _, row in df_final.iterrows():
    axes[1].annotate(row["ticker"],
                     (row["mean_ensemble"], row["latest_1d_return"] * 100),
                     textcoords="offset points", xytext=(6, 4), fontsize=9)
axes[1].axhline(0, color="black", linewidth=0.6, linestyle="--")
axes[1].axvline(0, color="black", linewidth=0.6, linestyle="--")
axes[1].set_title("Ensemble sentiment vs latest 1-day return (%)")
axes[1].set_xlabel("Mean ensemble score")
axes[1].set_ylabel("Latest 1-day return (%)")

plt.tight_layout()
plt.show()

print("\n── Final report table ──")
print(df_final.to_string(index=False))

In [ ]:
## ── Cell 23 · Export results ─────────────────────────────────────────────────
import pathlib

OUT_DIR = pathlib.Path("/content/output") if IN_COLAB else pathlib.Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_full.to_csv(OUT_DIR / "annotated_live_news.csv", index=False)
df_final.to_csv(OUT_DIR / "ensemble_vs_returns.csv", index=False)
df_pipeline.to_csv(OUT_DIR / "pipeline_normalizer_scores.csv", index=False)

print(f"Results saved to {OUT_DIR.resolve()}")
print(f"  annotated_live_news.csv   — {len(df_full)} rows")
print(f"  ensemble_vs_returns.csv  — {len(df_final)} rows")
print(f"  pipeline_normalizer_scores.csv — {len(df_pipeline)} rows")

---
## Workflow helper — fix → commit → push → restart

Run this cell **after local fixes** to push your changes to GitHub, then restart the kernel and re-run from the failed cell.

In [ ]:
## ── Cell 24 · Fix → commit → push → restart (working paradigm) ──────────────
# Usage on Colab:
#   1. Edit the source files locally (or in `files` panel on Colab)
#   2. Fill in COMMIT_MSG below
#   3. Run this cell
#   4. Go to Runtime → Restart session
#   5. Re-run from the failing cell

COMMIT_MSG = "fix: <describe your fix here>"  # <-- edit before running

if IN_COLAB:
    REPO_DIR = "/content/sentvols_repo"
    import subprocess as _sp

    def _run(cmd, **kwargs):
        result = _sp.run(cmd, shell=True, capture_output=True, text=True,
                         cwd=REPO_DIR, **kwargs)
        output = (result.stdout + result.stderr).strip()
        if output:
            print(output)
        if result.returncode != 0:
            raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
        return result

    _run("git config user.email 'colab@sentvols.ci'")
    _run("git config user.name  'Colab CI'")

    # Stage all changes (source + notebook)
    _run("git add -A")

    # Only commit if there is something staged
    status = _sp.run("git status --porcelain", shell=True, capture_output=True,
                     text=True, cwd=REPO_DIR).stdout.strip()
    if status:
        _run(f'git commit -m "{COMMIT_MSG}"')
        _run("git push origin main")
        print("\nPushed. Now go to Runtime → Restart session and re-run from the failing cell.")
    else:
        print("Nothing to commit.")
else:
    print("Not running in Colab — use your local git workflow:")
    print("  git add -A && git commit -m 'fix: ...' && git push origin main")